In [0]:
%run ../00-common/01.environment-config

In [0]:
control_path = "s3://bizf1project123/Control/Aaisha_increment/batch_control"

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id").strip()

if not v_batch_id:
    raise ValueError("batch_id is missing")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

In [0]:
delta_table = DeltaTable.forPath(spark, control_path)

source_df = (
    spark.createDataFrame([(v_batch_id,)], ["batch_id"])
    .withColumn("status", F.lit("completed"))
    .withColumn("updated_timestamp", F.current_timestamp())
)

(
    delta_table.alias("t")
    .merge(
        source_df.alias("s"),
        "t.batch_id = s.batch_id AND t.status = 'in_progress'"
    )
    .whenMatchedUpdate(set={
        "status": "s.status",
        "updated_timestamp": "s.updated_timestamp"
    })
    .execute()
)

print(f"Marked batch {v_batch_id} as completed")

In [0]:
display(
    spark.read.format("delta")
    .load(control_path)
    .orderBy("batch_id")
)